# My First LangGraph Workflow: BMI Calculator

This notebook demonstrates a simple **BMI Calculation Workflow** using LangGraph. 
It consists of two main nodes:
1. **Calculate BMI**: Computes the BMI from weight (kg) and height (m).
2. **Categorize BMI**: Determines the weight category (Normal, Overweight, etc.).

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# 1. Define the State
class BMIState(TypedDict):
    weight_kg: float
    height_m: float
    bmi: float
    category: str

# 2. Define the Node Functions
def calculate_bmi(state: BMIState):
    """Calculates BMI from weight and height in the state."""
    bmi = state['weight_kg'] / (state['height_m'] ** 2)
    print(f"--- Node: Calculate BMI ---")
    print(f"Inputs: Weight={state['weight_kg']}kg, Height={state['height_m']}m")
    print(f"Result: {round(bmi, 2)}")
    return {"bmi": round(bmi, 2)}

def categorize_bmi(state: BMIState):
    """Categorizes the calculated BMI."""
    bmi = state['bmi']
    if bmi < 18.5:
        category = "Underweight"
    elif bmi < 25:
        category = "Normal"
    elif bmi < 30:
        category = "Overweight"
    else:
        category = "Obese"
    
    print(f"\n--- Node: Categorize BMI ---")
    print(f"Input BMI: {bmi}")
    print(f"Category: {category}")
    return {"category": category}

# 3. Build the Graph
builder = StateGraph(BMIState)

# Add nodes
builder.add_node("calculate", calculate_bmi)
builder.add_node("categorize", categorize_bmi)

# Add edges defining the flow: START -> calculate -> categorize -> END
builder.add_edge(START, "calculate")
builder.add_edge("calculate", "categorize")
builder.add_edge("categorize", END)

# 4. Compile the Graph
graph = builder.compile()

# 5. Execute the Workflow
input_data = {
    "weight_kg": 75,
    "height_m": 1.80
}

final_result = graph.invoke(input_data)

print("\n--- Final Workflow Result ---")
print(f"Final State: {final_result}")